In [1]:
import requests
import time
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD

# API configuration
API_KEY  = "27d4364fba3f7caee94dcd212a4b7199"
BASE_URL = "https://api.themoviedb.org/3"

# Define namespaces
MV  = Namespace("http://moviekg.local/resource/")   # Entities
MVO = Namespace("http://moviekg.local/ontology/")   # Ontology (classes and properties)

In [4]:
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Fetch movie overview from TMDB
def fetch_movie_overview(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY, "language": "en-US"}
    r = requests.get(url, params=params, timeout=10)
    data = r.json()
    return data.get("title", ""), data.get("overview", "")

# Test NER on a few movies
test_movies = [27205, 157336, 603, 550, 238]  # Inception, Interstellar, Matrix, Fight Club, Godfather

print("NER Results \n")
for movie_id in test_movies:
    title, overview = fetch_movie_overview(movie_id)
    if not overview:
        continue
    
    doc = nlp(overview)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    
    print(f"Movie: {title}")
    print(f"Overview: {overview[:150]}...")
    print(f"Entities: {entities}")

NER Results 

Movie: Inception
Overview: Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as pa...
Entities: []
Movie: Interstellar
Overview: The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the va...
Entities: []
Movie: The Matrix
Overview: Set in the 22nd century, The Matrix tells the story of a computer hacker who joins a group of underground insurgents fighting the vast and powerful co...
Entities: [('the 22nd century', 'DATE')]
Movie: Fight Club
Overview: A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches o...
Entities: []
Movie: The Godfather
Overview: Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito 

In [5]:
# 定义本体结构：类和属性
# 包含Movie、Person、Genre、ProductionCompany四个类
# 以及hasDirector、hasActor、hasGenre等关系属性
def build_ontology(g):
    # Classes
    for cls in ["Movie", "Person", "Genre", "ProductionCompany"]:
        g.add((MVO[cls], RDF.type, OWL.Class))

    # Properties
    props = {
        "hasDirector":        ("Movie",   "Person"),
        "hasActor":           ("Movie",   "Person"),
        "hasGenre":           ("Movie",   "Genre"),
        "producedBy":         ("Movie",   "ProductionCompany"),
        "releaseYear":        ("Movie",   None),   # datatype property
        "hasTitle":           ("Movie",   None),
    }
    for prop, (domain, range_) in props.items():
        g.add((MVO[prop], RDF.type, OWL.ObjectProperty))
        g.add((MVO[prop], RDFS.domain, MVO[domain]))
        if range_:
            g.add((MVO[prop], RDFS.range, MVO[range_]))
    
    return g

In [6]:
# 将任意字符串转换为合法的URI片段
# 去除空格、斜杠、引号等特殊字符，避免URI格式错误
def safe_uri(text):
    """Convert a string into a valid URI fragment"""
    return (str(text).strip()
            .replace(" ", "_")
            .replace("/", "-")
            .replace("'", "")
            .replace(",", "")
            .replace(".", ""))
    
# 从TMDB API获取热门电影列表
# 每页返回20部电影，page参数控制页码
def fetch_movie_details(movie_id):
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {
        "api_key": API_KEY,
        "language": "en-US",
        "append_to_response": "credits"  # Fetch both actors and directors
    }
    r = requests.get(url, params=params, timeout=10)
    return r.json()

# 根据电影ID获取详细信息，包括演员和导演
# append_to_response=credits 同时获取credits信息，减少API调用次数
def fetch_popular_movies(page=1):
    url = f"{BASE_URL}/movie/popular"
    params = {"api_key": API_KEY, "language": "en-US", "page": page}
    r = requests.get(url, params=params, timeout=10)
    return r.json().get("results", [])

In [7]:
# 将单部电影的详细信息转换为RDF三元组并加入图中
# 包括：基本信息、类型、制作公司、导演、演员（前5名）
def add_movie_triples(g, detail):
    movie_id   = detail["id"]
    title      = detail.get("title", "Unknown")
    movie_uri  = MV[f"movie_{movie_id}"]

    # Basic information
    g.add((movie_uri, RDF.type,       MVO.Movie))
    g.add((movie_uri, MVO.hasTitle,   Literal(title, datatype=XSD.string)))

    year = detail.get("release_date", "")[:4]
    if year:
        g.add((movie_uri, MVO.releaseYear, Literal(int(year), datatype=XSD.integer)))

    # Genres
    for genre in detail.get("genres", []):
        genre_uri = MV[safe_uri(genre["name"])]
        g.add((genre_uri, RDF.type, MVO.Genre))
        g.add((movie_uri, MVO.hasGenre, genre_uri))

    # Production companies
    for comp in detail.get("production_companies", []):
        comp_uri = MV[f"company_{comp['id']}"]
        g.add((comp_uri, RDF.type, MVO.ProductionCompany))
        g.add((comp_uri, RDFS.label, Literal(comp["name"])))
        g.add((movie_uri, MVO.producedBy, comp_uri))

    credits = detail.get("credits", {})

    # Directors
    for crew in credits.get("crew", []):
        if crew["job"] == "Director":
            person_uri = MV[f"person_{crew['id']}"]
            g.add((person_uri, RDF.type, MVO.Person))
            g.add((person_uri, RDFS.label, Literal(crew["name"])))
            g.add((movie_uri, MVO.hasDirector, person_uri))

    # Actors (first 5)
    for cast in credits.get("cast", [])[:5]:
        person_uri = MV[f"person_{cast['id']}"]
        g.add((person_uri, RDF.type, MVO.Person))
        g.add((person_uri, RDFS.label, Literal(cast["name"])))
        g.add((movie_uri, MVO.hasActor, person_uri))


# Main program
# 爬取10页热门电影（共200部），构建RDF知识图谱
# 每次请求间隔0.25秒，避免触发API限流
# 最终保存为movie_kb.ttl文件
g = Graph()
g.bind("mv", MV); g.bind("mvo", MVO)
g = build_ontology(g)

for page in range(1, 11):          # 10 pages × 20 movies = 200 movies
    for movie in fetch_popular_movies(page):
        detail = fetch_movie_details(movie["id"])
        add_movie_triples(g, detail)
        time.sleep(0.25)           # Avoid triggering API rate limits
    print(f"Page {page} done, triples so far: {len(g)}")

g.serialize("movie_kb.ttl", format="turtle")
print(f"Done! Total triples: {len(g)}")

Page 1 done, triples so far: 691
Page 2 done, triples so far: 1242
Page 3 done, triples so far: 1823
Page 4 done, triples so far: 2046
Page 5 done, triples so far: 2659
Page 6 done, triples so far: 3230
Page 7 done, triples so far: 3802
Page 8 done, triples so far: 4002
Page 9 done, triples so far: 4538
Page 10 done, triples so far: 4806
Done! Total triples: 4806
